# ctgkit — CSV string → Polars → 30-min epoch → analysis → plot

This notebook shows an end-to-end flow:

1. **Ingest a CSV *string*** (as if it arrived from an API/message queue) that carries
   **wall-clock timestamps** and is **longer than 30 minutes**.
2. Use **Polars** to inspect how much time the recording covers, **cast the timestamp
   column to seconds-from-start**, **infer the sampling rate from the timestamps**, and
   **window to the most recent 30 minutes**.
3. Feed the windowed arrays into `ctgkit` and run the analysis.
4. Render the clinician-style trace.

> The only timing information we assume to have is **a timestamp per measurement** —
> there is no declared sampling rate. We derive `hz` from the spacing between
> timestamps. ctgkit aligns samples to a uniform time base, so we also sanity-check
> that the spacing is regular (and note what to do if it isn't).

In [ ]:
import io
from datetime import datetime, timedelta

import numpy as np
import polars as pl

import ctgkit
from ctgkit.synth import synth_epoch   # only used to fabricate a realistic demo signal

EXPECTED_MIN = 30.0
TOLERANCE_MIN = 2.0   # the "+ what is allowed to add" band: only trim if > 32 min

# NOTE: GEN_HZ is used *only* to fabricate the demo CSV below (a stand-in for a
# real monitor export). The ingestion code further down never references it —
# the sampling rate is inferred purely from the timestamps in the file.
GEN_HZ = 4.0

## 1. Build a CSV string with real timestamps, longer than 30 min

We fabricate a **45-minute** recording (with late decelerations, so the analysis is
interesting) and serialise it to CSV text with an ISO-8601 `timestamp` column. Treat
`csv_text` below as opaque input that arrived from somewhere external — **all we get is
a timestamp per row.**

In [ ]:
# --- fabricate a 45-min signal and emit it as a CSV STRING ---
demo = synth_epoch("late_decels", minutes=45.0, hz=GEN_HZ, seed=7)
start = datetime(2024, 6, 1, 9, 0, 0)

rows = ["timestamp,fhr,toco"]
for i, (f, u) in enumerate(zip(demo.fhr, demo.toco)):
    ts = (start + timedelta(seconds=i / GEN_HZ)).isoformat()
    fhr_cell = "" if np.isnan(f) else f"{f:.2f}"   # blanks model signal loss
    rows.append(f"{ts},{fhr_cell},{u:.2f}")
csv_text = "\n".join(rows)

print(f"{len(csv_text):,} chars, {csv_text.count(chr(10)) + 1:,} lines")
print("\n".join(csv_text.splitlines()[:4]))

## 2. Read the string with Polars and check how much time it covers

`try_parse_dates=True` turns the `timestamp` column into a real `Datetime`, so we can
measure the recording's span directly.

In [ ]:
df = pl.read_csv(io.StringIO(csv_text), try_parse_dates=True)
print("dtypes:", dict(zip(df.columns, df.dtypes)))

span = df["timestamp"].max() - df["timestamp"].min()
total_min = span.total_seconds() / 60.0
print(f"recording covers {total_min:.2f} min  ({df.height:,} samples)")
df.head(3)

## 3. Cast to seconds-from-start and infer the sampling rate

ctgkit works in seconds relative to the start of the epoch, so we subtract the first
timestamp. Since no sampling rate is given, we **derive `hz` from the timestamps**: the
median gap between consecutive samples is the sampling period, and `hz = 1 / period`.

We use the **median** gap (robust to occasional dropouts) and check that the spacing is
regular. ctgkit assumes a uniform time base — if your source is genuinely irregular,
resample onto a fixed grid before this step (e.g. `df.upsample(...)` /
`group_by_dynamic` in Polars) instead of trusting a single inferred rate.

In [ ]:
df = df.with_columns(
    (
        (pl.col("timestamp") - pl.col("timestamp").first())
        .dt.total_microseconds() / 1_000_000
    ).alias("time_s")
)

# infer sampling rate from the spacing between timestamps
gaps = df["time_s"].diff().drop_nulls()
period_s = float(gaps.median())
inferred_hz = 1.0 / period_s
rel_spread = float(gaps.std() / gaps.mean())   # ~0 means evenly spaced

print(f"median sample period : {period_s:.4f} s")
print(f"inferred sampling rate: {inferred_hz:.3f} Hz")
print(f"spacing regularity    : {rel_spread:.2%} relative spread "
      f"({'regular' if rel_spread < 0.05 else 'IRREGULAR — consider resampling'})")

df.select(["timestamp", "time_s", "fhr", "toco"]).head(3)

## 4. Window to the most recent 30 minutes

Only trim if the recording exceeds `EXPECTED_MIN + TOLERANCE_MIN` (32 min) — anything
already inside the contract is left untouched. This step is purely time-based (it uses
`time_s`, not the sample rate). Filtering rows in Polars is cheap, and because we trim
**before** handing data to ctgkit, the signal-processing only ever runs on a 30-minute
window regardless of how long the original file was.

In [ ]:
last_s = df["time_s"][-1]
total_min = last_s / 60.0

if total_min > EXPECTED_MIN + TOLERANCE_MIN:
    cutoff_s = last_s - EXPECTED_MIN * 60.0
    windowed = df.filter(pl.col("time_s") >= cutoff_s)
    print(f"trimmed {total_min:.1f} min -> most recent {EXPECTED_MIN:.0f} min")
else:
    windowed = df
    print(f"{total_min:.1f} min already within contract; no trim")

win_min = (windowed["time_s"][-1] - windowed["time_s"][0]) / 60.0
print(f"window: {win_min:.2f} min  ({windowed.height:,} samples)")

## 5. Hand the arrays to ctgkit and analyse

We pull the windowed columns out as NumPy arrays and build a `Signal` with
`from_arrays()`, passing the **inferred** `hz`. Then `analyze()` returns the category,
alert, and structured concerns.

In [ ]:
signal = ctgkit.from_arrays(
    fhr=windowed["fhr"].to_numpy(),
    toco=windowed["toco"].to_numpy(),
    hz=inferred_hz,                      # derived from the timestamps, not assumed
    meta={"source": "csv_string",
          "inferred_hz": round(inferred_hz, 3),
          "windowed_from_min": round(total_min, 1)},
)

result = ctgkit.analyze(signal, guideline="figo", metadata={"oxytocin": True})
print(result.summary())

In [ ]:
# Fully JSON-serialisable, audit-ready output
import json
print(json.dumps(result.to_dict(), indent=2)[:900], "...")

## 6. Plot the trace

`ctgkit.plot()` returns a matplotlib `Figure` (the alert banner, baseline, normal band,
and flagged concern windows are overlaid). Returning it as the last expression renders
it inline.

In [ ]:
%matplotlib inline
fig = ctgkit.plot(signal, result)
fig